##### Imputation Based on Missing Percentages: 

Different imputation methods were applied based on the percentage of missing values for each group (combination of `Client`, `Warehouse`, and `Product`)
- Low Missing Percentage (< 30%): linear interpolation
- Moderate Missing Percentage (30-70%): Iterative Imputer
- High Missing Percentage (> 70%): Local Median Imputation
- High Missing Percentage: Global median Imputation

In [10]:
# Import necessary libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm  # Import tqdm for the progress bar
from sklearn.experimental import enable_iterative_imputer  # Enable Iterative Imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

In [11]:
# ----------------------------------------------------
# Step 1: Data Loading and Setup
# ----------------------------------------------------
## Define file paths for the different phases
# Get the current working directory
dir_path = "/home/ubuntu/anup/projects/enterprise_forecasting/"

# Append the parent directory to sys.path
sys.path.append(os.path.join(dir_path, "./"))

# File path to the data file
raw_file_path = os.path.join(dir_path, 'data', "phase_1_raw_data.parquet")  
print(raw_file_path)

# Read the data 
raw_df = pd.read_parquet(raw_file_path)

# Convert all Prices which are zero to NaN for interpolation 
raw_df['Price'] = raw_df['Price'].replace(0, np.nan)

# Define thresholds for missing percentages
low_threshold = 30  # Less than 30% missing
moderate_threshold = 70  # 30-70% missing

# Step 1: Calculate the percentage of missing values for each combination of Client, Warehouse, and Product
missing_stats = raw_df.groupby(['Client', 'Warehouse', 'Product'])['Price'].apply(lambda x: x.isnull().mean() * 100).reset_index(name='MissingPercentage')

/home/ubuntu/anup/projects/enterprise_forecasting/data/phase_1_raw_data.parquet


In [12]:
# ----------------------------------------------------
# Step 2: Impute for low missing percentages (< 30%) using linear interpolation
# ----------------------------------------------------

# Filter combinations with less than 30% missing values
low_missing = missing_stats[missing_stats['MissingPercentage'] < low_threshold]

# Apply linear interpolation for each combination
for _, row in low_missing.iterrows():
    client, warehouse, product = row['Client'], row['Warehouse'], row['Product']
    
    # Filter rows for the specific combination
    mask = (raw_df['Client'] == client) & (raw_df['Warehouse'] == warehouse) & (raw_df['Product'] == product)
    
    # Apply linear interpolation within the group
    raw_df.loc[mask, 'Price'] = raw_df.loc[mask, 'Price'].interpolate(method='linear')

In [13]:
# ----------------------------------------------------
# Step 3: Impute for moderate missing percentages (30-80%) using Iterative Imputer
# ----------------------------------------------------

# Filter combinations with 30-80% missing values
moderate_missing = missing_stats[(missing_stats['MissingPercentage'] >= low_threshold) & (missing_stats['MissingPercentage'] <= moderate_threshold)]

# Encode categorical values for use in Iterative Imputer
encoder = OrdinalEncoder()
encoded_data = raw_df.copy()
encoded_data[['Client', 'Warehouse', 'Product']] = encoder.fit_transform(encoded_data[['Client', 'Warehouse', 'Product']])

# Filter the data for moderate missing combinations
moderate_data = encoded_data[(encoded_data['Client'].isin(moderate_missing['Client'])) &
                             (encoded_data['Warehouse'].isin(moderate_missing['Warehouse'])) &
                             (encoded_data['Product'].isin(moderate_missing['Product']))]

# Apply Iterative Imputer
imputer = IterativeImputer(max_iter=10, random_state=0)

# Fit and transform the moderate missing data
moderate_imputed = imputer.fit_transform(moderate_data[['Client', 'Warehouse', 'Product', 'Price', 'y']])

# Update the original dataframe with the imputed Price values
encoded_data.loc[moderate_data.index, 'Price'] = moderate_imputed[:, 3]  # Column index 3 corresponds to 'Price'


In [14]:
# ----------------------------------------------------
# Step 4: Impute for high missing percentages (> 80%) using local median
# ----------------------------------------------------

# Filter combinations with more than 80% missing values
high_missing = missing_stats[missing_stats['MissingPercentage'] > moderate_threshold]

# Step 4.1: First, impute missing values using the local median from Phase 1
for _, row in high_missing.iterrows():
    client, warehouse, product = row['Client'], row['Warehouse'], row['Product']
    
    # Filter rows for the specific combination in Phase 1
    mask_phase_1 = (raw_df['Client'] == client) & (raw_df['Warehouse'] == warehouse) & (raw_df['Product'] == product) & (raw_df['Phase'] == 'Phase 1')
    
    # Calculate the local median for Phase 1 for the combination and fill missing values
    phase_1_median = raw_df.loc[mask_phase_1, 'Price'].median()
    raw_df.loc[mask_phase_1, 'Price'] = raw_df.loc[mask_phase_1, 'Price'].fillna(phase_1_median)

# Step 4.2: For any remaining missing values, impute using the local median from Phase 0
for _, row in high_missing.iterrows():
    client, warehouse, product = row['Client'], row['Warehouse'], row['Product']
    
    # Filter rows for the specific combination in Phase 0
    mask_phase_0 = (raw_df['Client'] == client) & (raw_df['Warehouse'] == warehouse) & (raw_df['Product'] == product) & (raw_df['Phase'] == 'Phase 0')
    
    # Calculate the local median for Phase 0 for the combination and fill missing values
    phase_0_median = raw_df.loc[mask_phase_0, 'Price'].median()
    raw_df.loc[mask_phase_0, 'Price'] = raw_df.loc[mask_phase_0, 'Price'].fillna(phase_0_median)

In [15]:
# ----------------------------------------------------
# Step 5: Impute any remaining missing values using the global median
# ----------------------------------------------------

# Calculate the global median for Price
global_median = raw_df['Price'].median()

# Impute any remaining missing values in Price with the global median
raw_df['Price'] = raw_df['Price'].fillna(global_median)


In [16]:
# ----------------------------------------------------
# Step 6: Verify if there are any remaining missing values in the Price column
# ----------------------------------------------------

remaining_missing_prices_count = raw_df['Price'].isnull().sum()
print(f"Number of missing values in 'Price' column after imputation: {remaining_missing_prices_count}")

# Check the updated dataframe to verify the imputation
print(raw_df[['Client', 'Warehouse', 'Product', 'ds', 'Price', 'y']].tail())


Number of missing values in 'Price' column after imputation: 0
         Client  Warehouse  Product         ds      Price    y
2754694      46        318    14294 2023-12-04  46.990002  0.0
2754695      46        318    14294 2023-12-11  46.990002  1.0
2754696      46        318    14294 2023-12-18  46.990002  1.0
2754697      46        318    14294 2023-12-25  39.189999  1.0
2754698      46        318    14294 2024-01-01  45.423336  3.0


In [17]:
# ----------------------------------------------------
# Optional: Check remaining missing combinations if needed
# ----------------------------------------------------

# Group by Client, Warehouse, and Product to see the remaining missing values for Price
remaining_missing = raw_df[raw_df['Price'].isnull()]

remaining_missing_stats = remaining_missing.groupby(['Client', 'Warehouse', 'Product']).size().reset_index(name='MissingCount')

# Sort by the number of missing values to see the most affected combinations
remaining_missing_stats = remaining_missing_stats.sort_values(by='MissingCount', ascending=False)

print("Combinations with remaining missing Price values:")
print(remaining_missing_stats.head(10))  # Show the top 10 combinations with missing prices

Combinations with remaining missing Price values:
Empty DataFrame
Columns: [Client, Warehouse, Product, MissingCount]
Index: []


In [18]:
unique_id = '4_203_11186'
check_df = raw_df[(raw_df['client_warehouse_product_id'] == unique_id)]
check_df.tail(15)

,Client,Warehouse,Product,ds,Price,y,client_warehouse_product_id,Phase
403068,4,203,11186,2023-09-25,30.0,0.0,4_203_11186,Phase 0
403069,4,203,11186,2023-10-02,30.0,0.0,4_203_11186,Phase 0
2589820,4,203,11186,2023-10-09,30.0,0.0,4_203_11186,Phase 1
2589821,4,203,11186,2023-10-16,30.0,0.0,4_203_11186,Phase 1
2589822,4,203,11186,2023-10-23,30.0,0.0,4_203_11186,Phase 1
2589823,4,203,11186,2023-10-30,30.0,0.0,4_203_11186,Phase 1
2589824,4,203,11186,2023-11-06,30.0,0.0,4_203_11186,Phase 1
2589825,4,203,11186,2023-11-13,30.0,0.0,4_203_11186,Phase 1
2589826,4,203,11186,2023-11-20,11.7,1.0,4_203_11186,Phase 1
2589827,4,203,11186,2023-11-27,30.0,0.0,4_203_11186,Phase 1


In [19]:
# Export the raw data to parquet file 
raw_df.to_parquet(os.path.join(dir_path, 'data', "phase_1_clean_data.parquet"))